In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'qiskit==2.2.3','qiskit-machine-learning==0.9.0','numpy==2.2.6',
    'scipy==1.15.3','scikit-learn==1.6.1','pandas==2.2.3',
    'matplotlib==3.10.1','flask==3.1.0'])

0

In [16]:
import numpy as np
x=np.array([5.5,.2,.8,10.])
conditions=np.array([x[0]<6,x[1]<.25,x[2]>.7])
print('조건별 결과:',conditions.tolist())
print('교육용 라벨:',int(conditions.any()))

조건별 결과: [True, True, True]
교육용 라벨: 1


In [15]:
import time
from pathlib import Path
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from qiskit.circuit.library import z_feature_map, zz_feature_map, pauli_feature_map
from qiskit.circuit.library import real_amplitudes, efficient_su2, n_local, pauli_two_design
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.primitives import QMLSampler, QMLEstimator
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import VQC

def parity(integer):
    return int(integer).bit_count() % 2

def circuits(n, fmap_name='Z', ansatz_name='RealAmplitudes', reps=1):
    if fmap_name == 'Z':
        fmap = z_feature_map(n, reps=1)
    elif fmap_name == 'ZZ':
        fmap = zz_feature_map(n, reps=1, entanglement='linear')
    elif fmap_name == 'Pauli':
        fmap = pauli_feature_map(n, reps=1, paulis=['Z','XX'], entanglement='linear')
    else:
        raise ValueError(fmap_name)
    if ansatz_name == 'RealAmplitudes':
        ansatz = real_amplitudes(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'EfficientSU2':
        ansatz = efficient_su2(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'TwoLocal':
        ansatz = n_local(n, rotation_blocks=['ry','rz'], entanglement_blocks='cz', reps=reps, entanglement='linear')
    elif ansatz_name == 'PauliTwoDesign':
        ansatz = pauli_two_design(n, reps=reps, seed=33)
    else:
        raise ValueError(ansatz_name)
    return fmap, ansatz

def fit_model(kind, train_x, train_y, fmap_name='Z', ansatz_name='RealAmplitudes', maxiter=40, reps=1):
    fmap, ansatz = circuits(train_x.shape[1], fmap_name, ansatz_name, reps)
    initial = np.linspace(0.1,0.7,ansatz.num_parameters)
    history = []
    def optimizer(fun, x0, jac=None, bounds=None):
        def tracked(weights):
            value = float(fun(weights))
            history.append(value)
            return value
        return minimize(tracked, x0, method='COBYLA', options={'maxiter':maxiter, 'rhobeg':0.3})
    started = time.perf_counter()
    if kind == 'VQC':
        model = VQC(feature_map=fmap, ansatz=ansatz, sampler=QMLSampler(shots=None),
                    interpret=parity, output_shape=2, initial_point=initial, optimizer=optimizer)
        model.fit(train_x, train_y)
        def predict_probability(x):
            return np.asarray(model.neural_network.forward(x, model.weights))[:,1]
        weights = model.weights
    else:
        circuit = fmap.compose(ansatz)
        if kind == 'SQNN':
            qnn = SamplerQNN(circuit=circuit, sampler=QMLSampler(shots=None),
                             input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                             interpret=parity, output_shape=2)
            def probability(x,w):
                return np.asarray(qnn.forward(x,w))[:,1]
            start = initial
        elif kind == 'EQNN':
            qnn = EstimatorQNN(circuit=circuit, estimator=QMLEstimator(default_precision=0),
                              observables=SparsePauliOp('Z'*train_x.shape[1]),
                              input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                              default_precision=0)
            def probability(x,w):
                expectation = np.asarray(qnn.forward(x,w[:-2])).ravel()
                # 마지막 두 값은 고전 출력층의 가중치와 편향입니다.
                score = np.clip(w[-2]*expectation+w[-1],-40,40)
                return 1/(1+np.exp(-score))
            start = np.r_[initial,-1.0,0.0]
        else:
            raise ValueError(kind)
        def loss(w):
            p1 = np.clip(probability(train_x,w),1e-10,1-1e-10)
            value = float(-np.mean(train_y*np.log2(p1)+(1-train_y)*np.log2(1-p1)))
            return value
        result = optimizer(loss,start)
        weights = result.x
        def predict_probability(x):
            return probability(x,weights)
        model = None
    print(kind, fmap_name, ansatz_name, '학습 평가 수:',len(history),
          '최초/마지막 기록 손실:',round(history[0],6),round(history[-1],6))
    return predict_probability, model, {'parameters':len(weights),'depth':fmap.compose(ansatz).depth(),
                                       'seconds':time.perf_counter()-started,'history':history}

def report(name,y,p1):
    p1 = np.asarray(p1).ravel()
    assert p1.shape == y.shape and np.isfinite(p1).all()
    assert np.all((p1>=-1e-8)&(p1<=1+1e-8))
    predicted = (p1>=0.5).astype(int)
    result = {'model':name, 'accuracy':accuracy_score(y,predicted),
              'precision':precision_score(y,predicted,zero_division=0),
              'recall':recall_score(y,predicted,zero_division=0),
              'f1':f1_score(y,predicted,zero_division=0)}
    print({k:round(float(v),4) if k!='model' else v for k,v in result.items()})
    print('혼동행렬 [[TN,FP],[FN,TP]]:',confusion_matrix(y,predicted,labels=[0,1]).tolist())
    return result

def split_fixed(X,y):
    # 각 클래스에서 원래 행 순서의 네 번째마다 시험 표본으로 사용합니다.
    test = np.concatenate([np.flatnonzero(y==c)[3::4] for c in [0,1]])
    train = np.setdiff1d(np.arange(len(y)),test)
    assert not np.intersect1d(train,test).size
    assert len(np.unique(y[train]))==len(np.unique(y[test]))==2
    return train,test

print('버전:',{k:metadata.version(k) for k in ['qiskit','qiskit-machine-learning','numpy','scipy','scikit-learn']})

X = np.array([[8.0, 0.4, 0.2, 40.0],
 [5.0, 0.2, 0.8, 10.0],
 [7.0, 0.3, 0.4, 30.0],
 [5.5, 0.3, 0.6, 20.0],
 [9.0, 0.5, 0.1, 50.0],
 [7.0, 0.2, 0.5, 25.0],
 [6.5, 0.3, 0.5, 20.0],
 [8.0, 0.4, 0.8, 30.0],
 [7.5, 0.35, 0.3, 35.0],
 [4.0, 0.15, 0.9, 0.0],
 [8.5, 0.45, 0.2, 45.0],
 [6.0, 0.2, 0.6, 15.0],
 [6.0, 0.25, 0.7, 10.0],
 [5.8, 0.4, 0.3, 40.0],
 [7.0, 0.5, 0.4, 55.0],
 [7.5, 0.2, 0.2, 50.0],
 [10.0, 0.6, 0.1, 60.0],
 [9.0, 0.5, 0.9, 60.0],
 [6.8, 0.28, 0.6, 15.0],
 [5.0, 0.35, 0.4, 30.0],
 [8.0, 0.3, 0.5, 0.0],
 [7.0, 0.24, 0.5, 20.0],
 [7.2, 0.32, 0.65, 25.0],
 [6.5, 0.3, 0.75, 35.0]], dtype=float)
y = ((X[:,0]<6)|(X[:,1]<.25)|(X[:,2]>.7)).astype(int)
train,test = split_fixed(X,y)
scaler = MinMaxScaler(feature_range=(0,1),clip=True).fit(X[train])
train_x,test_x = scaler.transform(X[train]),scaler.transform(X[test])
print('학습/시험 크기:',train_x.shape,test_x.shape)
print('학습/시험 라벨 수:',np.bincount(y[train]).tolist(),np.bincount(y[test]).tolist())
predict,model,info=fit_model('VQC',train_x,y[train],'Z','PauliTwoDesign',maxiter=80)
report('SleepVQC',y[test],predict(test_x))
sample=np.array([[5.5,.2,.8,10]])
p1=float(predict(scaler.transform(sample))[0])
print('단건 P(1)/예측:',round(p1,6),int(p1>=.5))
print('시험 정답:',y[test].tolist())
print('시험 P(1):',np.round(predict(test_x),4).tolist())

버전: {'qiskit': '2.2.3', 'qiskit-machine-learning': '0.9.0', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1'}
학습/시험 크기: (18, 4) (6, 4)
학습/시험 라벨 수: [9, 9] [3, 3]
VQC Z PauliTwoDesign 학습 평가 수: 80 최초/마지막 기록 손실: 0.964994 0.910774
{'model': 'SleepVQC', 'accuracy': 0.5, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6667}
혼동행렬 [[TN,FP],[FN,TP]]: [[0, 3], [0, 3]]
단건 P(1)/예측: 0.790186 1
시험 정답: [0, 0, 0, 1, 1, 1]
시험 P(1): [0.6458, 0.6091, 0.6446, 0.574, 0.5544, 0.7237]


시험 혼동행렬 [[0,3],[0,3]]은 여섯 기록을 모두 라벨 1로 분류한 결과입니다. 라벨 1의 재현율은 1.0이고 정밀도는 0.5, 정확도는 0.5, F1은 약 0.6667입니다. 정상 라벨 3건이 모두 오탐으로 분류됐음을 함께 확인해야 합니다. 이 결과는 해당 설정에서 두 그룹을 충분히 구분하지 못했음을 보여줍니다.

단건 입력 [5.5,0.2,0.8,10]의 모델 확률은 약 0.790186입니다. 코드가 반환한 실제 결과를 사용해 학습 예산·입력 특징·회로 구조를 실험합니다. 실제 수면 상태의 의학적 판단은 이 규칙 분류의 평가 범위에 포함되지 않습니다.